Scripts for *Function-Calling and Data Extraction with LLMs*, in order, with the output from my run. Scripts 03 to 07 need Ollama with `granite4.2:3b`, and they read saved replies from a local cache when one exists. To run a cell yourself, start Jupyter in the `scripts/` folder, because the scripts look for their data one folder up.

## 2. What the model sees: a tool definition is text

In [1]:
# What a model actually reads when you give it tools
# pip install transformers
#
# Tool definitions are not sent through a separate channel. The chat template
# writes them into the system prompt as text, so they cost tokens on every request.
import textwrap
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")


def make_tool_definition(name, description, parameter, parameter_description):
    """One tool definition in the JSON shape chat APIs use."""
    return {"type": "function", "function": {
        "name": name,
        "description": description,
        "parameters": {
            "type": "object",
            "properties": {parameter: {"type": "string", "description": parameter_description}},
            "required": [parameter],
        },
    }}


tools = [
    make_tool_definition("fetch_arxiv", "Fetch the text of a paper from arXiv by its ID.", "arxiv_id", "The arXiv ID, for example 2408.02442"),
    make_tool_definition("get_section", "Return one named section of the paper already fetched.", "name", "Section title, for example Results"),
    make_tool_definition("search_arxiv", "Search arXiv and return titles with their IDs.", "query", "Search words"),
    make_tool_definition("list_sections", "List the section titles of the fetched paper.", "arxiv_id", "The arXiv ID"),
    make_tool_definition("get_references", "Return the reference list of the fetched paper.", "arxiv_id", "The arXiv ID"),
    make_tool_definition("run_python", "Run a short Python snippet and return its output.", "code", "Python source code"),
    make_tool_definition("read_file", "Read a text file from the lab folder.", "path", "Relative file path"),
    make_tool_definition("write_file", "Write a text file into the lab folder.", "path", "Relative file path"),
    make_tool_definition("list_files", "List the files in the lab folder.", "folder", "Relative folder path"),
    make_tool_definition("get_abstract", "Return only the abstract of a paper.", "arxiv_id", "The arXiv ID"),
]

messages = [{"role": "user", "content": "Get paper 2408.02442"}]


def count_prompt_tokens(tool_count):
    template_options = {"tools": tools[:tool_count]} if tool_count else {}
    text = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False, **template_options)
    return len(tokenizer(text).input_ids), text


tokens_without_tools, _ = count_prompt_tokens(0)
print(f"{'tools':>5}  {'prompt tokens':>13}  {'added by the tools':>18}")
for tool_count in (0, 1, 2, 5, 10):
    token_count, _ = count_prompt_tokens(tool_count)
    print(f"{tool_count:>5}  {token_count:>13}  {token_count - tokens_without_tools:>18}")  # tokens beyond the no-tool prompt

# The system block the model really reads when one tool is given
_, text = count_prompt_tokens(1)
print()
for line in text.split("<|im_start|>user")[0].splitlines():  # wrapped so it fits a narrow page
    print(textwrap.fill(line, 76, subsequent_indent="  "))

tools  prompt tokens  added by the tools
    0             42                   0
    1            212                 170
    2            283                 241
    5            498                 456
   10            838                 796

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.

# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"type": "function", "function": {"name": "fetch_arxiv", "description":
  "Fetch the text of a paper from arXiv by its ID.", "parameters": {"type":
  "object", "properties": {"arxiv_id": {"type": "string", "description":
  "The arXiv ID, for example 2408.02442"}}, "required": ["arxiv_id"]}}}
</tools>

For each function call, return a json object with function name and
  arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <function-name>, "arguments": <args-json-object>}
</tool_cal

## 1. The model asks, your code runs: the two tools

In [1]:
# The two tools the Reader will use, written by hand
# pip install requests pypdf
#
# fetch_arxiv downloads a paper once and keeps its text in a local cache.
# get_section returns one section of it. If the name is wrong, the error text
# lists the real section names, so a model can correct itself.
import re
import sys
import time
from pathlib import Path

import requests
from pypdf import PdfReader

ARGS = sys.argv[1:] if "__file__" in globals() else []  # a notebook passes its own arguments, ignore them
SCRIPT_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()  # a notebook has no __file__

CACHE = SCRIPT_DIR.parent / ".paper_cache"
ARXIV_ID = re.compile(r"^\d{4}\.\d{4,5}(v\d+)?$")
# "1 Introduction", "4.2 Results", plus the unnumbered headings papers use
# Section numbers start at 1 ("0.42 Ounce" is body text) and headings have no brackets
HEADING = re.compile(r"^((?!0\.)\d{1,2}(\.\d{1,2})*\.?\s+[A-Z][^\n()]{2,70}|Abstract|References|Appendix[^\n]{0,40})$")

PAPERS = ["2310.13548", "2408.02442", "2501.10868", "2303.11366", "2303.17651",
          "2302.04761", "2210.03629", "2201.11903", "1706.03762", "2307.03172"]


def fetch_arxiv(arxiv_id: str) -> str:
    """Return the paper's text. Downloads on the first call, then reads the cache."""
    if not ARXIV_ID.match(arxiv_id):
        return f"Error: '{arxiv_id}' is not an arXiv ID like 2408.02442."
    CACHE.mkdir(exist_ok=True)
    cached = CACHE / f"{arxiv_id}.txt"
    if cached.exists():
        return cached.read_text()
    pdf = CACHE / f"{arxiv_id}.pdf"
    response = requests.get(f"https://arxiv.org/pdf/{arxiv_id}", timeout=60,
                        headers={"User-Agent": "paper-lab/0.1 (personal learning project)"})
    if response.status_code != 200:
        return f"Error: arXiv answered {response.status_code} for {arxiv_id}."
    pdf.write_bytes(response.content)
    text = "\n".join(page.extract_text() or "" for page in PdfReader(pdf).pages)
    cached.write_text(text)
    pdf.unlink()  # keep only the text
    time.sleep(3)  # arXiv asks automated clients to go slowly
    return text


def split_into_sections(text: str) -> dict:
    """Cut the text at lines that look like headings."""
    sections, name, lines = {}, "Start", []
    for line in text.splitlines():
        line = line.strip()
        if HEADING.match(line):
            sections[name] = "\n".join(lines).strip()
            name, lines = line, []
        else:
            lines.append(line)
    sections[name] = "\n".join(lines).strip()
    return {heading: body for heading, body in sections.items() if body}


def normalize_heading(heading: str) -> str:
    """Lowercase and drop everything but letters and digits: 'I NTRODUCTION' -> 'introduction'."""
    return re.sub(r"[^a-z0-9]", "", heading.lower())


def get_section(arxiv_id: str, name: str) -> str:
    """Return the first section whose heading contains `name`, ignoring case and spacing."""
    text = fetch_arxiv(arxiv_id)
    if text.startswith("Error:"):
        return text
    sections = split_into_sections(text)
    for heading, body in sections.items():
        if normalize_heading(name) in normalize_heading(heading):
            return body
    return f"Error: no section called '{name}'. Sections: {', '.join(sections)}"


if __name__ == "__main__":
    arxiv_ids = ARGS or PAPERS
    print(f"{'paper':<11} {'chars':>7}  {'sections':>8}")
    for arxiv_id in arxiv_ids:
        text = fetch_arxiv(arxiv_id)
        if text.startswith("Error:"):
            print(f"{arxiv_id:<11} {text}")
        else:
            print(f"{arxiv_id:<11} {len(text):>7}  {len(split_into_sections(text)):>8}")

paper         chars  sections
2310.13548   115194        18
2408.02442    56560        23
2501.10868    69515        21
2303.11366    59439        16
2303.17651   124985        14
2302.04761    71962        24
2210.03629   110119        10
2201.11903   135737        20
1706.03762    39510        24
2307.03172    65453        23


## 3. Asking for a form: schema, validation and retry

In [1]:
# The Reader: claims from a paper as validated JSON
# pip install requests pydantic     (and run Ollama: ollama pull granite4.2:3b)
#
# The model gets the abstract and a description of the JSON shape. Python checks
# the reply against a schema, retries with the error shown (capped), and then
# scores the claims against a hand-written reference list.
import hashlib
import json
import re
import sys
import time
import unicodedata
from pathlib import Path

import requests
from pydantic import BaseModel, ValidationError

ARGS = sys.argv[1:] if "__file__" in globals() else []  # a notebook passes its own arguments, ignore them
SCRIPT_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()  # a notebook has no __file__

HERE = SCRIPT_DIR.parent
MODEL = "granite4.2:3b"
MAX_RETRIES = 2  # the cap: first try plus two retries, then the paper counts as failed

SYSTEM = ("You extract testable claims from a paper. Reply with JSON only, shaped like "
          '{"claims": [{"claim": "...", "how_to_test": "...", "expected_result": "..."}]}. '
          "Put exact numbers from the text into claim and expected_result.")


class Claim(BaseModel):
    claim: str
    how_to_test: str
    expected_result: str


class Claims(BaseModel):
    claims: list[Claim]


def ask_ollama(messages, temperature=0.0, seed=0, num_ctx=8192, model=None):
    """One call to a local Ollama model. Every answer is cached on disk."""
    payload = {"model": model or MODEL, "messages": messages, "stream": False, "keep_alive": "10m",
               "options": {"temperature": temperature, "seed": seed, "num_ctx": num_ctx}}
    key = hashlib.sha256(("v2" + json.dumps(payload, sort_keys=True)).encode()).hexdigest()[:20]
    cache = HERE / ".model_cache" / f"{key}.json"
    if cache.exists():
        return json.loads(cache.read_text())
    started = time.time()
    response = requests.post("http://localhost:11434/api/chat", json=payload, timeout=900)
    if response.status_code != 200:  # loud: show Ollama's own reason, not just the status code
        raise RuntimeError(f"Ollama answered {response.status_code}: {response.text[:300]}")
    data = response.json()
    result = {"text": data["message"]["content"], "thinking_chars": len(data["message"].get("thinking") or ""),
           "tokens_in": data.get("prompt_eval_count", 0),
           "tokens_out": data.get("eval_count", 0), "secs": round(time.time() - started, 1)}
    cache.parent.mkdir(exist_ok=True)
    cache.write_text(json.dumps(result))
    return result


# In these PDFs a figure or footnote sits between the abstract and the introduction
ABSTRACT_ENDS = {"2201.11903": "A: The cafeteria", "1706.03762": "∗Equal contribution"}


def get_abstract(arxiv_id):
    """Abstract text from the paper cached by 02_fetch_arxiv_by_hand.py."""
    # NFKC turns PDF ligatures such as the single "fi" glyph back into plain letters
    text = unicodedata.normalize("NFKC", (HERE / ".paper_cache" / f"{arxiv_id}.txt").read_text())
    match = re.search(r"abstract", text, re.I)
    rest = text[match.end():] if match else text
    end = re.search(r"\n\s*1\.?\s+(I\s?NTRODUCTION|Introduction)", rest)
    stop = end.start() if end else 3000
    marker = rest.find(ABSTRACT_ENDS.get(arxiv_id, "\0"))
    if marker != -1:
        stop = min(stop, marker)
    return re.sub(r"\s+", " ", rest[:stop])[:3000]


def parse_claims(reply):
    """Return (claims, error). Models often wrap JSON in a code fence, so strip that."""
    cleaned = re.sub(r"^```(?:json)?|```$", "", reply.strip(), flags=re.M).strip()
    try:
        return Claims.model_validate_json(cleaned), None
    except (ValidationError, ValueError) as problem:
        return None, str(problem).splitlines()[0][:160]


def extract_claims(text, temperature=0.0, seed=0, num_ctx=8192):
    """Ask, validate, retry with the error shown, give up after MAX_RETRIES."""
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": text}]
    stats = {"tokens_in": 0, "tokens_out": 0, "secs": 0.0, "tries": 0}
    for attempt in range(MAX_RETRIES + 1):
        result = ask_ollama(messages, temperature, seed, num_ctx)
        for field in ("tokens_in", "tokens_out", "secs"):
            stats[field] += result[field]
        stats["tries"] = attempt + 1
        claims, error = parse_claims(result["text"])
        if claims:
            return claims, stats
        messages += [{"role": "assistant", "content": result["text"]},
                     {"role": "user", "content": f"That was not valid: {error}. Reply with corrected JSON only."}]
    return None, stats  # failed: never invent a result


def score_claims(claims, references):
    """How many reference claims have one extracted claim matching every pattern."""
    claim_texts = [(claim.claim + " " + claim.expected_result) for claim in claims.claims] if claims else []
    return [any(all(re.search(pattern, text, re.I) for pattern in reference["facts"]) for text in claim_texts)
            for reference in references]


if __name__ == "__main__":
    reference_claims = json.loads((HERE / "reference_claims.json").read_text())
    reference_claims = {arxiv_id: claims for arxiv_id, claims in reference_claims.items() if not arxiv_id.startswith("_")}
    model = ARGS[0] if ARGS else MODEL
    MODEL = model
    rows, found, total = [], 0, 0
    print(f"{'paper':<11} {'tries':>5} {'valid':>5} {'found':>5} {'tokens in':>9} {'tokens out':>10} {'seconds':>7}")
    for arxiv_id, references in reference_claims.items():
        claims, stats = extract_claims(get_abstract(arxiv_id))
        hits = score_claims(claims, references)
        found, total = found + sum(hits), total + len(hits)
        print(f"{arxiv_id:<11} {stats['tries']:>5} {'yes' if claims else 'NO':>5} {sum(hits):>3}/{len(hits)} "
              f"{stats['tokens_in']:>9} {stats['tokens_out']:>10} {stats['secs']:>7.0f}")
        rows.append({"paper": arxiv_id, "valid": bool(claims), "hits": hits, **stats})
    print(f"\nclaims found: {found}/{total}  valid JSON: {sum(row['valid'] for row in rows)}/{len(rows)}")
    (HERE / "results").mkdir(exist_ok=True)
    (HERE / "results" / "03_extract.json").write_text(json.dumps({"model": model, "rows": rows}, indent=1))

paper       tries valid found tokens in tokens out seconds
2310.13548      1   yes   3/3       299       1881      37
2408.02442      1   yes   1/3       230       1676      31
2501.10868      1   yes   3/3       366       2150      40
2303.11366      1   yes   1/3       361       1652      31
2303.17651      1   yes   3/3       328       1599      29
2302.04761      1   yes   1/3       282       3275      62
2210.03629      1   yes   2/3       370       2370      44
2201.11903      1   yes   1/3       236       2048      38
1706.03762      1   yes   3/3       298       1448      26
2307.03172      1   yes   3/3       250       1492      27

claims found: 21/30  valid JSON: 10/10


## 5. Temperature: the same prompt, different answers

In [1]:
# Temperature: the same extraction five times at 0 and at 1
# pip install requests pydantic     (and run Ollama with granite4.2:3b)
#
# Each run uses a different seed. At temperature 0 the seed should not matter,
# so the five answers should match. At temperature 1 every run is a new spin.
# Needs the abstracts cached by 02_fetch_arxiv_by_hand.py.
#
#   python3 04_temperature.py            reads the saved same-seed counts (fast)
#   python3 04_temperature.py --fresh    makes the 10 same-seed calls again (slow, may differ)
import hashlib
import json
import re
import sys
import time
import unicodedata
from collections import Counter
from pathlib import Path

import requests
from pydantic import BaseModel, ValidationError

ARGS = sys.argv[1:] if "__file__" in globals() else []  # a notebook passes its own arguments, ignore them
SCRIPT_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()  # a notebook has no __file__

HERE = SCRIPT_DIR.parent
MODEL = "granite4.2:3b"
MAX_RETRIES = 2
PAPERS = ["1706.03762", "2210.03629", "2303.11366"]  # all found, partly found, mostly missed in script 03
SEEDS = [1, 2, 3, 4, 5]
ABSTRACT_ENDS = {"1706.03762": "∗Equal contribution"}

SYSTEM = ("You extract testable claims from a paper. Reply with JSON only, shaped like "
          '{"claims": [{"claim": "...", "how_to_test": "...", "expected_result": "..."}]}. '
          "Put exact numbers from the text into claim and expected_result.")


class Claim(BaseModel):
    claim: str
    how_to_test: str
    expected_result: str


class Claims(BaseModel):
    claims: list[Claim]


def ask_ollama(messages, temperature, seed, use_cache=True):
    """One call to a local Ollama model, cached on disk by its full request."""
    payload = {"model": MODEL, "messages": messages, "stream": False, "keep_alive": "10m",
               "options": {"temperature": temperature, "seed": seed, "num_ctx": 8192}}
    key = hashlib.sha256(("v2" + json.dumps(payload, sort_keys=True)).encode()).hexdigest()[:20]
    cache = HERE / ".model_cache" / f"{key}.json"
    if use_cache and cache.exists():
        return json.loads(cache.read_text())
    started = time.time()
    response = requests.post("http://localhost:11434/api/chat", json=payload, timeout=900)
    if response.status_code != 200:  # loud: show Ollama's own reason, not just the status code
        raise RuntimeError(f"Ollama answered {response.status_code}: {response.text[:300]}")
    data = response.json()
    result = {"text": data["message"]["content"], "thinking_chars": len(data["message"].get("thinking") or ""),
           "tokens_in": data.get("prompt_eval_count", 0), "tokens_out": data.get("eval_count", 0),
           "secs": round(time.time() - started, 1)}
    if use_cache:
        cache.parent.mkdir(exist_ok=True)
        cache.write_text(json.dumps(result))
    return result


def get_abstract(arxiv_id):
    text = unicodedata.normalize("NFKC", (HERE / ".paper_cache" / f"{arxiv_id}.txt").read_text())
    match = re.search(r"abstract", text, re.I)
    rest = text[match.end():] if match else text
    end = re.search(r"\n\s*1\.?\s+(I\s?NTRODUCTION|Introduction)", rest)
    stop = end.start() if end else 3000
    marker = rest.find(ABSTRACT_ENDS.get(arxiv_id, "\0"))
    if marker != -1:
        stop = min(stop, marker)
    return re.sub(r"\s+", " ", rest[:stop])[:3000]


def parse_claims(reply):
    cleaned = re.sub(r"^```(?:json)?|```$", "", reply.strip(), flags=re.M).strip()
    try:
        return Claims.model_validate_json(cleaned), None
    except (ValidationError, ValueError) as problem:
        return None, str(problem).splitlines()[0][:160]


def extract_claims(text, temperature, seed):
    """Ask once; if the JSON is bad, retry with the error shown, up to MAX_RETRIES."""
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": text}]
    first_reply = ask_ollama(messages, temperature, seed)
    result, tokens, tries = first_reply, first_reply["tokens_out"], 1
    for _ in range(MAX_RETRIES + 1):
        claims, error = parse_claims(result["text"])
        if claims or tries > MAX_RETRIES:
            return claims, first_reply, tokens, tries
        messages += [{"role": "assistant", "content": result["text"]},
                     {"role": "user", "content": f"That was not valid: {error}. Reply with corrected JSON only."}]
        result = ask_ollama(messages, temperature, seed)
        tokens, tries = tokens + result["tokens_out"], tries + 1
    return None, first_reply, tokens, tries


def matched_reference_positions(claims, references):
    """Which reference claims (by position) one extracted claim matches."""
    claim_texts = [(claim.claim + " " + claim.expected_result) for claim in claims.claims] if claims else []
    return tuple(position for position, reference in enumerate(references)
                 if any(all(re.search(pattern, text, re.I) for pattern in reference["facts"]) for text in claim_texts))


if __name__ == "__main__":
    reference_claims = json.loads((HERE / "reference_claims.json").read_text())
    FRESH = "--fresh" in ARGS  # repeat the slow same-seed check instead of reading its saved counts
    papers = [argument for argument in ARGS if not argument.startswith("--")] or PAPERS
    rows = []
    print(f"{'paper':<11} {'temp':>4} {'valid':>5} {'found':>5} {'agree':>5} {'distinct':>8} {'tokens out':>10}")
    for arxiv_id in papers:
        for temperature in (0.0, 1.0):
            runs = [extract_claims(get_abstract(arxiv_id), temperature, seed) for seed in SEEDS]
            found_sets = [matched_reference_positions(claims, reference_claims[arxiv_id]) for claims, _, _, _ in runs]
            first_try = sum(tries == 1 and claims is not None for claims, _, _, tries in runs)
            agree = Counter(found_sets).most_common(1)[0][1]  # runs that match the most common result
            distinct = len({first_reply["text"] for _, first_reply, _, _ in runs})  # how many different first replies
            mean_found = sum(len(found) for found in found_sets) / len(found_sets)
            mean_tokens = sum(tokens for _, _, tokens, _ in runs) / len(runs)
            print(f"{arxiv_id:<11} {temperature:>4.0f} {first_try:>3}/5 {mean_found:>5.1f} {agree:>3}/5 {distinct:>6}/5 {mean_tokens:>10.0f}")
            rows.append({"paper": arxiv_id, "temp": temperature, "valid_first_try": first_try, "mean_found": mean_found,
                         "agree": agree, "distinct": distinct, "mean_tokens_out": mean_tokens,
                         "sets": [list(found) for found in found_sets]})
    # Does temperature 0 repeat itself? Same prompt, same seed, five fresh calls, cache off.
    # Slow (10 calls) and not repeatable by design, so a rerun reads the counts saved by the
    # last fresh run. Add --fresh to run the calls again.
    print("\nsame seed, five fresh calls at temperature 0")
    saved = HERE / "results" / "04_temperature.json"
    same = json.loads(saved.read_text()).get("same_seed_distinct") if saved.exists() and not FRESH else None
    if same:
        for arxiv_id, distinct_replies in same.items():
            print(f"{arxiv_id:<11} distinct replies: {distinct_replies}/5   (saved from the last fresh run)")
    else:
        same = {}
        for arxiv_id in ("2210.03629", "2303.11366"):
            messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": get_abstract(arxiv_id)}]
            texts = [ask_ollama(messages, 0.0, 1, use_cache=False)["text"] for _ in range(5)]
            same[arxiv_id] = len(set(texts))
            print(f"{arxiv_id:<11} distinct replies: {same[arxiv_id]}/5")
    (HERE / "results").mkdir(exist_ok=True)
    (HERE / "results" / "04_temperature.json").write_text(json.dumps({"model": MODEL, "rows": rows, "same_seed_distinct": same}, indent=1))

paper       temp valid found agree distinct tokens out
1706.03762     0   5/5   3.0   5/5      1/5       1448
1706.03762     1   5/5   2.8   4/5      5/5       2299
2210.03629     0   5/5   2.0   5/5      2/5       2795
2210.03629     1   5/5   2.0   5/5      5/5       2622
2303.11366     0   5/5   1.8   4/5      2/5       1715
2303.11366     1   5/5   1.8   4/5      5/5       2464

same seed, five fresh calls at temperature 0
2210.03629  distinct replies: 2/5   (saved from the last fresh run)
2303.11366  distinct replies: 2/5   (saved from the last fresh run)


## 6. The context window: what shares it

In [1]:
# Context window: the whole paper at once, or one section at a time
# pip install requests pydantic     (and run Ollama with granite4.2:3b)
#
# Three ways to read the same paper:
#   small window  whole paper, Ollama's usual 4096-token window (Ollama refuses it, loudly)
#   big window    whole paper, a 20480-token window
#   sections      one call per section, claims merged afterwards
# Section calls may write at most MAX_OUT tokens (thinking included). A call that hits
# the cap is counted and shown in the table, never hidden.
# Needs the papers cached by 02_fetch_arxiv_by_hand.py.
import hashlib
import json
import re
import sys
import time
import unicodedata
from pathlib import Path

import requests
from pydantic import BaseModel, ValidationError

ARGS = sys.argv[1:] if "__file__" in globals() else []  # a notebook passes its own arguments, ignore them
SCRIPT_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()  # a notebook has no __file__

HERE = SCRIPT_DIR.parent
MODEL = "granite4.2:3b"
PAPERS = ["1706.03762", "2303.11366"]
SMALL, BIG = 4096, 20480
MAX_RETRIES = 2  # per chunk, then the chunk counts as invalid
MAX_OUT = 4000  # cap for the section calls
HEADING = re.compile(r"^((?!0\.)\d{1,2}(\.\d{1,2})*\.?\s+[A-Z][^\n()]{2,70}|Abstract|References|Appendix[^\n]{0,40})$")

SYSTEM = ("You extract testable claims from paper text. Reply with JSON only, shaped like "
          '{"claims": [{"claim": "...", "how_to_test": "...", "expected_result": "..."}]}. '
          "Give at most 5 claims, the most important first. Put exact numbers from the text into "
          "claim and expected_result.")


class Claim(BaseModel):
    claim: str
    how_to_test: str
    expected_result: str


class Claims(BaseModel):
    claims: list[Claim]


def show_progress(text):
    print("    " + text, file=sys.stderr, flush=True)


def ask_ollama(messages, num_ctx, max_out=None, call_label=""):
    """One call to a local Ollama model at temperature 0, cached on disk."""
    payload = {"model": MODEL, "messages": messages, "stream": False, "keep_alive": "10m",
               "options": {"temperature": 0, "seed": 1, "num_ctx": num_ctx}}
    if max_out:
        payload["options"]["num_predict"] = max_out  # stop a runaway call
    key = hashlib.sha256(("v2" + json.dumps(payload, sort_keys=True)).encode()).hexdigest()[:20]
    cache = HERE / ".model_cache" / f"{key}.json"
    if cache.exists():
        result = json.loads(cache.read_text())
        show_progress(f"{call_label}: from cache ({result['tokens_in']} tokens in, {result['tokens_out']} tokens out, {result['secs']} s when it ran)")
        return result
    show_progress(f"{call_label}: asking Ollama (window {num_ctx}, cap {max_out or 'none'}) ...")
    started = time.time()
    try:
        response = requests.post("http://localhost:11434/api/chat", json=payload, timeout=600)
    except requests.exceptions.Timeout:  # a stuck call must not freeze the whole run
        show_progress(f"{call_label}: TIMED OUT after 600 s")
        return {"text": "", "error": "no answer after 600 s", "thinking_chars": 0,
                "tokens_in": 0, "tokens_out": 0, "secs": 600, "capped": False}
    if response.status_code == 400:  # Ollama refused the request: keep its reason as a result
        show_progress(f"{call_label}: REFUSED by Ollama after {time.time() - started:.0f} s")
        result = {"text": "", "error": response.text[:300], "thinking_chars": 0, "tokens_in": 0, "tokens_out": 0, "secs": 0, "capped": False}
        return result  # not cached, so a fixed setup is retried next time
    if response.status_code != 200:  # loud: show Ollama's own reason, not just the status code
        raise RuntimeError(f"Ollama answered {response.status_code}: {response.text[:300]}")
    data = response.json()
    result = {"text": data["message"]["content"], "thinking_chars": len(data["message"].get("thinking") or ""),
           "tokens_in": data.get("prompt_eval_count", 0), "tokens_out": data.get("eval_count", 0),
           "secs": round(time.time() - started, 1), "capped": data.get("done_reason") == "length"}
    cache.parent.mkdir(exist_ok=True)
    cache.write_text(json.dumps(result))
    show_progress(f"{call_label}: done in {result['secs']:.0f} s ({result['tokens_in']} tokens in, {result['tokens_out']} tokens out, "
        f"thinking {result['thinking_chars']} characters" + (", HIT THE CAP" if result["capped"] else "") + ")")
    return result


def load_paper_text(arxiv_id):
    raw = (HERE / ".paper_cache" / f"{arxiv_id}.txt").read_text()
    return unicodedata.normalize("NFKC", raw)


def split_paper_into_chunks(text, min_chars=1500, max_chars=12000):
    """Cut at headings, glue short sections to the next one, drop references."""
    parts, name, lines = [], "start", []
    for line in text.splitlines():
        if HEADING.match(line.strip()):
            parts.append((name, "\n".join(lines)))
            name, lines = line.strip(), []
        else:
            lines.append(line)
    parts.append((name, "\n".join(lines)))
    chunks, buffer = [], ""
    for name, body in parts:
        if name.lower().startswith(("references", "appendix")):
            break
        buffer += f"\n{name}\n{body}"
        if len(buffer) >= min_chars:
            chunks.append(buffer[:max_chars])
            buffer = ""
    if buffer.strip():
        chunks.append(buffer[:max_chars])
    return [re.sub(r"[ \t]+", " ", chunk).strip() for chunk in chunks]


def parse_claims(reply):
    """Return (claims, error). Models often wrap JSON in a code fence, so strip that."""
    cleaned = re.sub(r"^```(?:json)?|```$", "", reply.strip(), flags=re.M).strip()
    try:
        return Claims.model_validate_json(cleaned).claims, None
    except (ValidationError, ValueError) as problem:
        return None, " ".join(str(problem).split())[:200]


def extract_claims_from_chunk(chunk, num_ctx, max_out, call_label=""):
    """One chunk: ask, validate, retry with the error shown (capped). A reply cut off by the
    token cap is not retried, because asking again would run into the same cap."""
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": chunk}]
    totals = {"claims": [], "refused": "", "invalid": False, "capped": False, "retries": 0, "first_in": 0,
         "tokens_in": 0, "tokens_out": 0, "think": 0, "secs": 0}
    for attempt in range(MAX_RETRIES + 1):
        result = ask_ollama(messages, num_ctx, max_out, f"{call_label} try {attempt + 1}")
        if attempt == 0:
            totals["first_in"] = result["tokens_in"]
        totals["tokens_in"] += result["tokens_in"]; totals["tokens_out"] += result["tokens_out"]
        totals["think"] += result.get("thinking_chars", 0); totals["secs"] += result["secs"]
        if result.get("error"):
            totals["refused"] = result["error"]
            return totals
        claims, error = parse_claims(result["text"])
        if claims is not None:
            totals["claims"] = claims
            return totals
        totals["capped"] = totals["capped"] or bool(result.get("capped"))
        if result.get("capped") or attempt == MAX_RETRIES:
            break
        messages += [{"role": "assistant", "content": result["text"]},
                     {"role": "user", "content": f"That was not valid: {error}. Reply with corrected JSON only."}]
        totals["retries"] += 1
    totals["invalid"] = True  # nothing usable: counted and shown, never invented
    return totals


def run_reading_method(label, chunks, num_ctx, references, max_out=None):
    results = [extract_claims_from_chunk(chunk, num_ctx, max_out,
                   f"{label} {number}/{len(chunks)} ({len(chunk)} chars, '{' '.join(chunk.split())[:28]}...')")
               for number, chunk in enumerate(chunks, 1)]
    claims = [claim for result in results for claim in result["claims"]]
    claim_texts = [claim.claim + " " + claim.expected_result for claim in claims]
    found = sum(any(all(re.search(pattern, text, re.I) for pattern in reference["facts"]) for text in claim_texts)
                for reference in references)
    tokens_sent = sum(len(chunk) for chunk in chunks) / 4  # rough guess: 4 characters per token
    tokens_read = sum(result["first_in"] for result in results)
    errors = [result["refused"] for result in results if result["refused"]]
    return {"how": label, "calls": len(results), "error": errors[0] if errors else "", "found": found,
            "claims": len(claims),
            "tokens_in": sum(result["tokens_in"] for result in results),
            "tokens_out": sum(result["tokens_out"] for result in results),
            "secs": round(sum(result["secs"] for result in results)),
            "capped": sum(result["capped"] for result in results),
            "invalid": sum(result["invalid"] for result in results),
            "retries": sum(result["retries"] for result in results),
            "think_chars": sum(result["think"] for result in results),
            "cut": tokens_read < 0.8 * tokens_sent}  # Ollama read clearly fewer tokens than we sent


if __name__ == "__main__":
    reference_claims = json.loads((HERE / "reference_claims.json").read_text())
    rows = []
    # capped = calls stopped by the token cap, result of all calls made (4/13 = 4 of 13 calls),
    # bad = replies that failed validation,
    # think = characters of hidden reasoning
    for arxiv_id in ARGS or PAPERS:
        text = load_paper_text(arxiv_id)
        print(f"\npaper {arxiv_id}")
        print(f"{'how':<8} {'found':>5} {'claims':>6} {'tokens in':>9} {'tokens out':>10} {'seconds':>7} "
              f"{'capped':>6} {'bad':>5} {'retry':>5} {'think':>6}")
        for label, chunks, window_size in (("small", [text], SMALL), ("big", [text], BIG),
                                   ("sections", split_paper_into_chunks(text), 8192)):
            row = run_reading_method(label, chunks, window_size, reference_claims[arxiv_id], MAX_OUT if label == "sections" else None)
            rows.append({"paper": arxiv_id, **row})
            capped, bad = f"{row['capped']}/{row['calls']}", f"{row['invalid']}/{row['calls']}"
            print(f"{label:<8} {row['found']:>3}/3 {row['claims']:>6} {row['tokens_in']:>9} "
                  f"{row['tokens_out']:>10} {row['secs']:>7} {capped:>6} {bad:>5} {row['retries']:>5} {row['think_chars']:>6}")
            if row["error"]:  # keep Ollama's own reason, shortened to its message
                reason = re.search(r"request \(\d+ tokens\) exceeds[^\\\"]*", row["error"])
                print("  Ollama refused the call:")
                print("  " + (reason.group(0).split(", try")[0] if reason else row["error"][:70]))
    (HERE / "results").mkdir(exist_ok=True)
    (HERE / "results" / "05_context_window.json").write_text(json.dumps({"model": MODEL, "rows": rows}, indent=1))


paper 1706.03762
how      found claims tokens in tokens out seconds capped   bad retry  think
small      0/3      0         0          0       0    0/1   0/1     0      0
  Ollama refused the call:
  request (10696 tokens) exceeds the available context size (4096 tokens)
big        2/3      5     21860       4413     194    0/1   0/1     1  12704
sections   3/3     36      8531      36429     731   4/13  4/13     0 123578

paper 2303.11366
how      found claims tokens in tokens out seconds capped   bad retry  think
small      0/3      0         0          0       0    0/1   0/1     0      0
  Ollama refused the call:
  request (16639 tokens) exceeds the available context size (4096 tokens)
big        2/3      4     16639       4117     181    0/1   0/1     0  12951


sections   1/3     31      9805      23217     479    1/9   1/9     0  81840


## 1. The model asks, your code runs: the loop (thinking on, for section 7)

In [1]:
# Function calling by hand: the model asks, our code runs the tool
# pip install requests pydantic pypdf     (and run Ollama with granite4.2:3b)
#
# The model never runs anything. It writes a request ("call get_section with these
# arguments"), THIS script runs the Python function and sends the result back, and
# the loop repeats until the model answers in plain text or the step cap is reached.
#
# Task per paper: read the abstract with the tools, then answer with claims as JSON.
# Needs the papers cached by 02_fetch_arxiv_by_hand.py.
#
#   python3 06_function_calling_by_hand.py                 all 10 papers
#   python3 06_function_calling_by_hand.py 2201.11903      one paper
#   python3 06_function_calling_by_hand.py --think off     ask Ollama to skip hidden reasoning
import hashlib
import importlib.util
import json
import re
import sys
import time
from pathlib import Path

import requests
from pydantic import BaseModel, ValidationError

ARGS = sys.argv[1:] if "__file__" in globals() else []  # a notebook passes its own arguments, ignore them
SCRIPT_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()  # a notebook has no __file__

HERE = SCRIPT_DIR.parent
MODEL = "granite4.2:3b"
MAX_STEPS = 8        # model calls per paper, tool rounds and retries together
MAX_RETRIES = 2      # bad final JSON: ask again with the error shown, this many times
MAX_OUT = 6000       # token cap for one call, hidden reasoning included (3000 cut off a good answer)
SECTION_CHARS = 6000  # a tool result stays short so it cannot flood the context window
BAR = 8              # checkpoint: this many of 10 papers must succeed to stay with a 3B model

# Reuse the two hand-written tools from script 02 (its file name starts with a digit,
# so it cannot be imported the usual way)
sys.dont_write_bytecode = True  # do not leave a __pycache__ folder next to the scripts
_tools_spec = importlib.util.spec_from_file_location("tools02", (SCRIPT_DIR / "02_fetch_arxiv_by_hand.py"))
paper_tools = importlib.util.module_from_spec(_tools_spec)
_tools_spec.loader.exec_module(paper_tools)

SYSTEM = ("You are a research assistant with two tools. To read a paper: call fetch_arxiv_sections "
          "to see its sections, then call get_section to read one. Once you have read the abstract, "
          "stop calling tools and reply with JSON only, shaped like "
          '{"claims": [{"claim": "...", "how_to_test": "...", "expected_result": "..."}]}. '
          "Put exact numbers from the text into claim and expected_result.")

# What the model sees. Name, description and parameters are written into its prompt.
TOOLS = [
    {"type": "function", "function": {
        "name": "fetch_arxiv_sections",
        "description": "Download an arXiv paper and list its section names. Use this first, before reading a section.",
        "parameters": {"type": "object", "properties": {
            "arxiv_id": {"type": "string", "description": "arXiv ID such as 2408.02442"}},
            "required": ["arxiv_id"]}}},
    {"type": "function", "function": {
        "name": "get_section",
        "description": "Return the text of one section of an arXiv paper, for example Abstract or Introduction.",
        "parameters": {"type": "object", "properties": {
            "arxiv_id": {"type": "string", "description": "arXiv ID such as 2408.02442"},
            "name": {"type": "string", "description": "section name, for example Abstract"}},
            "required": ["arxiv_id", "name"]}}},
]


# ---- the tools themselves: ordinary Python, run by this script, never by the model ----
def fetch_arxiv_sections(arxiv_id: str) -> str:
    text = paper_tools.fetch_arxiv(arxiv_id)
    if text.startswith("Error:"):
        return text
    names = list(paper_tools.split_into_sections(text))
    return f"Paper {arxiv_id} fetched ({len(text)} characters). Sections: " + "; ".join(names[:30])


def get_section(arxiv_id: str, name: str) -> str:
    text = paper_tools.get_section(arxiv_id, name)
    return text[:SECTION_CHARS]


REGISTRY = {"fetch_arxiv_sections": fetch_arxiv_sections, "get_section": get_section}


def run_tool(name, args):
    """Run one requested call. Every failure comes back as text the model can read."""
    if name not in REGISTRY:
        return f"Error: there is no tool called '{name}'. Tools: {', '.join(REGISTRY)}.", False
    if isinstance(args, str):
        try:
            args = json.loads(args)
        except ValueError:
            return "Error: the arguments were not valid JSON.", False
    try:
        return REGISTRY[name](**args), True
    except TypeError as problem:  # wrong or missing argument names
        return f"Error: bad arguments for {name}: {problem}", False


class Claim(BaseModel):
    claim: str
    how_to_test: str
    expected_result: str


class Claims(BaseModel):
    claims: list[Claim]


def parse_claims(reply):
    cleaned = re.sub(r"^```(?:json)?|```$", "", reply.strip(), flags=re.M).strip()
    try:
        return Claims.model_validate_json(cleaned).claims, None
    except (ValidationError, ValueError) as problem:
        return None, " ".join(str(problem).split())[:200]


def show_progress(text):
    print("    " + text, file=sys.stderr, flush=True)


THINK = None  # set by --think off


def ask_ollama_with_tools(messages, call_label):
    """One call to Ollama with the tool definitions attached. Cached on disk."""
    payload = {"model": MODEL, "messages": messages, "tools": TOOLS, "stream": False, "keep_alive": "10m",
               "options": {"temperature": 0, "seed": 1, "num_ctx": 8192, "num_predict": MAX_OUT}}
    if THINK is not None:
        payload["think"] = THINK
    key = hashlib.sha256(("v1" + json.dumps(payload, sort_keys=True)).encode()).hexdigest()[:20]
    cache = HERE / ".model_cache" / f"{key}.json"
    if cache.exists():
        result = json.loads(cache.read_text())
        show_progress(f"{call_label}: from cache ({result['secs']} s when it ran)")
        return result
    show_progress(f"{call_label}: asking Ollama ...")
    started = time.time()
    try:
        response = requests.post("http://localhost:11434/api/chat", json=payload, timeout=600)
    except requests.exceptions.Timeout:
        show_progress(f"{call_label}: TIMED OUT after 600 s")
        return {"error": "no answer after 600 s", "message": {}, "tokens_in": 0, "tokens_out": 0, "secs": 600, "capped": False}
    if response.status_code != 200:  # loud: keep Ollama's reason, do not cache
        show_progress(f"{call_label}: REFUSED by Ollama ({response.status_code}): {response.text[:200]}")
        return {"error": response.text[:300], "message": {}, "tokens_in": 0, "tokens_out": 0, "secs": 0, "capped": False}
    data = response.json()
    message = data["message"]
    result = {"message": {"content": message.get("content", ""), "tool_calls": message.get("tool_calls") or [],
                       "thinking_chars": len(message.get("thinking") or "")},
           "tokens_in": data.get("prompt_eval_count", 0), "tokens_out": data.get("eval_count", 0),
           "secs": round(time.time() - started, 1), "capped": data.get("done_reason") == "length"}
    cache.parent.mkdir(exist_ok=True)
    cache.write_text(json.dumps(result))
    kind = "tool request" if result["message"]["tool_calls"] else "text answer"
    show_progress(f"{call_label}: done in {result['secs']:.0f} s, {kind} ({result['tokens_in']} tokens in, {result['tokens_out']} tokens out, "
        f"thinking {result['message']['thinking_chars']} characters" + (", HIT THE CAP" if result["capped"] else "") + ")")
    return result


def run_tool_loop(arxiv_id):
    """The loop. Returns a record of everything that happened."""
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": f"Read the abstract of arXiv paper {arxiv_id} with your tools, "
                                            "then reply with its testable claims as JSON."}]
    record = {"paper": arxiv_id, "calls": [], "tool_errors": 0, "retries": 0, "model_calls": 0, "tokens_in": 0,
              "tokens_out": 0, "secs": 0.0, "claims": [], "ended": "", "read_ok": False}
    for step in range(1, MAX_STEPS + 1):
        reply = ask_ollama_with_tools(messages, f"{arxiv_id} step {step}/{MAX_STEPS}")
        record["model_calls"] += 1
        record["tokens_in"] += reply["tokens_in"]; record["tokens_out"] += reply["tokens_out"]; record["secs"] += reply["secs"]
        if reply.get("error"):
            record["ended"] = "ollama error: " + reply["error"][:100]
            return record
        message = reply["message"]
        if message["tool_calls"]:
            messages.append({"role": "assistant", "content": message["content"], "tool_calls": message["tool_calls"]})
            for call in message["tool_calls"]:
                name, args = call["function"]["name"], call["function"].get("arguments", {})
                tool_result, ran_ok = run_tool(name, args)
                if name == "get_section" and ran_ok and not tool_result.startswith("Error:"):
                    record["read_ok"] = True
                if not ran_ok or tool_result.startswith("Error:"):
                    record["tool_errors"] += 1
                record["calls"].append({"tool": name, "args": args, "result_start": tool_result[:120]})
                show_progress(f"  -> code ran {name}({json.dumps(args)[:90]}) -> {tool_result[:70]!r}")
                messages.append({"role": "tool", "tool_name": name, "content": tool_result})
            continue
        claims, error = parse_claims(message["content"])
        if claims is not None:
            record["claims"], record["ended"] = claims, "answered"
            return record
        if reply["capped"]:
            record["ended"] = "answer cut off by the token cap"
            return record
        if record["retries"] >= MAX_RETRIES:
            record["ended"] = "invalid JSON after retries"
            return record
        record["retries"] += 1
        messages += [{"role": "assistant", "content": message["content"]},
                     {"role": "user", "content": f"That was not valid: {error}. Reply with corrected JSON only."}]
    record["ended"] = f"step cap ({MAX_STEPS}) reached"
    return record


def count_found_claims(claims, references):
    claim_texts = [claim.claim + " " + claim.expected_result for claim in claims]
    return sum(any(all(re.search(pattern, text, re.I) for pattern in reference["facts"]) for text in claim_texts)
               for reference in references)


if __name__ == "__main__":
    args = ARGS
    if "--think" in args:
        position = args.index("--think")
        THINK = args[position + 1] != "off"
        del args[position:position + 2]
    reference_claims = json.loads((HERE / "reference_claims.json").read_text())
    reference_claims = {arxiv_id: claims for arxiv_id, claims in reference_claims.items() if not arxiv_id.startswith("_")}
    papers = args or list(reference_claims)
    print(f"{'paper':<11} {'tool calls':<10} {'errs':>4} {'retry':>5} {'calls':>5} {'found':>5} {'tokens out':>10} {'seconds':>7}  ended")
    rows, successes = [], 0
    for arxiv_id in papers:
        record = run_tool_loop(arxiv_id)
        record["found"] = count_found_claims(record["claims"], reference_claims[arxiv_id])
        record["success"] = record["read_ok"] and record["ended"] == "answered"
        successes += record["success"]
        call_sequence = ">".join(call["tool"].replace("fetch_arxiv_sections", "fetch").replace("get_section", "get")
                                 for call in record["calls"]) or "none"
        status = "OK" if record["success"] else "FAIL"
        print(f"{arxiv_id:<11} {call_sequence[:10]:<10} {record['tool_errors']:>4} {record['retries']:>5} {record['model_calls']:>5} "
              f"{record['found']:>3}/3 {record['tokens_out']:>10} {record['secs']:>7.0f}  {status}")
        if not record["success"]:
            print(f"            {record['ended']}")
        rows.append({**record, "claims": [claim.model_dump() for claim in record["claims"]]})
    print(f"\ncheckpoint: {successes}/{len(papers)} tasks succeeded")
    print("(success = read a section with the tools AND answered in valid JSON)")
    print(f"bar for staying with a 3B model: {BAR}/10")
    (HERE / "results").mkdir(exist_ok=True)
    suffix = "" if THINK is None else ("_think_on" if THINK else "_think_off")  # keep each setting's results apart
    (HERE / "results" / f"06_function_calling_by_hand{suffix}.json").write_text(
        json.dumps({"model": MODEL, "think": THINK, "rows": rows}, indent=1, default=str))

paper       tool calls errs retry calls found tokens out seconds  ended
2310.13548  fetch>get     0     0     3   0/3       6274     148  FAIL
            answer cut off by the token cap
2408.02442  fetch>get     0     0     3   2/3       2387      48  OK
2501.10868  fetch>get     0     0     3   2/3       2388      47  OK
2303.11366  fetch>get     0     0     3   2/3       1834      35  OK
2303.17651  fetch>get     0     0     3   2/3       1844      37  OK
2302.04761  fetch>get     0     0     3   2/3       1867      41  OK
2210.03629  fetch>get     0     0     3   2/3       2839      56  OK
2201.11903  fetch>get     0     0     3   1/3       5846     128  OK
1706.03762  fetch>get     0     0     3   3/3       2322      46  OK
2307.03172  fetch>get     0     0     3   2/3       2304      44  OK

checkpoint: 9/10 tasks succeeded
(success = read a section with the tools AND answered in valid JSON)
bar for staying with a 3B model: 8/10


## 4. Three ways to get the form back

In [1]:
# The same claim extraction, three ways of getting a filled-in form back
# pip install requests pydantic langchain-ollama     (and run Ollama with granite4.2:3b)
#
#   hand-written                ask in the prompt, validate with Pydantic, retry with the
#                               error shown (script 03)
#   LangChain json_schema       with_structured_output(method="json_schema"): the schema goes
#                               to Ollama as `format`, so the model can only write text that fits
#   LangChain function_calling  with_structured_output(method="function_calling"): the schema is
#                               offered as a tool and the model "calls" it
#
# All three read the same abstracts with the same instructions, at temperature 0, thinking off.
# Needs the papers cached by 02_fetch_arxiv_by_hand.py.
# A method that returns nothing usable is counted as invalid and its reason is printed, never hidden.
#
#   python3 07_langchain_structured_output.py              all 10 papers
#   python3 07_langchain_structured_output.py 2201.11903   one paper
import hashlib
import importlib.util
import json
import sys
import textwrap
import time
from pathlib import Path

import requests

ARGS = sys.argv[1:] if "__file__" in globals() else []  # a notebook passes its own arguments, ignore them
SCRIPT_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()  # a notebook has no __file__

sys.dont_write_bytecode = True  # do not leave a __pycache__ folder next to the scripts
HERE = SCRIPT_DIR.parent
MODEL = "granite4.2:3b"
MAX_OUT = 4000  # token cap per call

# Reuse the schema, prompt, abstract finder, parser and scorer from script 03
_reader_spec = importlib.util.spec_from_file_location("s03", (SCRIPT_DIR / "03_extract_claims.py"))
reader = importlib.util.module_from_spec(_reader_spec)
_reader_spec.loader.exec_module(reader)
SYSTEM, Claims, MAX_RETRIES = reader.SYSTEM, reader.Claims, reader.MAX_RETRIES

OPTIONS = {"temperature": 0, "seed": 1, "num_ctx": 8192, "num_predict": MAX_OUT}


def show_progress(text):
    print("    " + text, file=sys.stderr, flush=True)


def read_cache(key):
    cache_file = HERE / ".model_cache" / f"{key}.json"
    return json.loads(cache_file.read_text()) if cache_file.exists() else None


def write_cache(key, value):
    cache_folder = HERE / ".model_cache"
    cache_folder.mkdir(exist_ok=True)
    (cache_folder / f"{key}.json").write_text(json.dumps(value))


def make_cache_key(*parts):
    return hashlib.sha256(json.dumps(parts, sort_keys=True).encode()).hexdigest()[:20]


# ---- arm 1: hand-written ----
def ask_ollama(messages, call_label):
    payload = {"model": MODEL, "messages": messages, "stream": False, "think": False,
               "keep_alive": "10m", "options": OPTIONS}
    key = make_cache_key("07-hand", payload)
    cached = read_cache(key)
    if cached:
        show_progress(f"{call_label}: from cache")
        return cached
    show_progress(f"{call_label}: asking Ollama ...")
    started = time.time()
    response = requests.post("http://localhost:11434/api/chat", json=payload, timeout=600)
    if response.status_code != 200:  # loud, not cached
        show_progress(f"{call_label}: REFUSED by Ollama ({response.status_code}): {response.text[:200]}")
        return {"text": "", "tokens_out": 0, "secs": 0, "error": response.text[:200]}
    data = response.json()
    result = {"text": data["message"]["content"], "tokens_out": data.get("eval_count", 0),
           "secs": round(time.time() - started, 1), "error": ""}
    write_cache(key, result)
    show_progress(f"{call_label}: done in {result['secs']:.0f} s ({result['tokens_out']} tokens out)")
    return result


def run_hand_written_method(abstract, arxiv_id):
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": abstract}]
    record = {"claims": None, "first_ok": False, "tries": 0, "tokens_out": 0, "secs": 0.0, "error": ""}
    for attempt in range(MAX_RETRIES + 1):
        result = ask_ollama(messages, f"{arxiv_id} hand-written try {attempt + 1}")
        record["tries"] = attempt + 1
        record["tokens_out"] += result["tokens_out"]
        record["secs"] += result["secs"]
        if result.get("error"):
            record["error"] = result["error"]
            return record
        claims, error = reader.parse_claims(result["text"])
        if claims:
            record["claims"], record["first_ok"] = claims, attempt == 0
            return record
        record["error"] = error
        messages += [{"role": "assistant", "content": result["text"]},
                     {"role": "user", "content": f"That was not valid: {error}. Reply with corrected JSON only."}]
    return record


# ---- arms 2 and 3: LangChain ----
def run_langchain_method(method, abstract, arxiv_id):
    from langchain_ollama import ChatOllama  # imported here so arm 1 works without LangChain

    call_label = f"{arxiv_id} lc {method}"
    # function_calling got a new cache version: its first run hid WHY a reply failed
    key = make_cache_key("07-lc2" if method == "function_calling" else "07-lc", method, SYSTEM, abstract, OPTIONS, MODEL)
    cached = read_cache(key)
    if cached:
        show_progress(f"{call_label}: from cache")
    else:
        show_progress(f"{call_label}: asking Ollama ...")
        llm = ChatOllama(model=MODEL, temperature=0, seed=1, num_ctx=8192, num_predict=MAX_OUT,
                         reasoning=False, keep_alive="10m")
        structured = llm.with_structured_output(Claims, method=method, include_raw=True)
        started = time.time()
        try:
            result = structured.invoke([("system", SYSTEM), ("human", abstract)])
            usage = getattr(result["raw"], "usage_metadata", None) or {}
            parsed = result["parsed"]
            error = " ".join(str(result["parsing_error"]).split())[:200] if result["parsing_error"] else ""
            if parsed is None and not error:
                # LangChain returns None and no error when the model never called the tool
                raw = result["raw"]
                error = (f"LangChain gave None and no error: the model made {len(raw.tool_calls or [])} tool call(s) "
                         f"and wrote: {' '.join((raw.content or '').split())[:150]!r}")
            cached = {"claims": parsed.model_dump() if parsed else None, "error": error,
                   "tokens_out": usage.get("output_tokens", 0), "secs": round(time.time() - started, 1)}
        except Exception as problem:  # for example Ollama refuses tools for this model: loud, not cached
            show_progress(f"{call_label}: FAILED before any reply: {type(problem).__name__}: {str(problem)[:160]}")
            return {"claims": None, "first_ok": False, "tries": 1, "tokens_out": 0, "secs": 0.0,
                    "error": f"{type(problem).__name__}: {str(problem)[:160]}"}
        write_cache(key, cached)
        show_progress(f"{call_label}: done in {cached['secs']:.0f} s ({cached['tokens_out']} tokens out)"
            + ("" if cached["claims"] else f", NO VALID RESULT: {cached['error'][:80]}"))
    claims = Claims.model_validate(cached["claims"]) if cached["claims"] else None
    return {"claims": claims, "first_ok": claims is not None, "tries": 1,
            "tokens_out": cached["tokens_out"], "secs": cached["secs"], "error": cached["error"]}


# The three methods compared. "LangChain" rows go through LangChain's with_structured_output.
METHODS = [("hand-written", run_hand_written_method),
           ("LangChain json_schema", lambda abstract, arxiv_id: run_langchain_method("json_schema", abstract, arxiv_id)),
           ("LangChain function_calling",
            lambda abstract, arxiv_id: run_langchain_method("function_calling", abstract, arxiv_id))]
# valid first try = the first reply passed validation; valid = it passed after any retries
HEAD = f"{'method':<26} {'valid first try':>15} {'valid':>5} {'found':>6} {'tokens out':>10} {'seconds':>7}"

if __name__ == "__main__":
    reference_claims = json.loads((HERE / "reference_claims.json").read_text())
    reference_claims = {arxiv_id: claims for arxiv_id, claims in reference_claims.items() if not arxiv_id.startswith("_")}
    papers = ARGS or list(reference_claims)
    rows = []
    for arxiv_id in papers:
        abstract = reader.get_abstract(arxiv_id)
        print(f"\npaper {arxiv_id}")
        print(HEAD)
        for name, run_method in METHODS:
            record = run_method(abstract, arxiv_id)
            found = sum(reader.score_claims(record["claims"], reference_claims[arxiv_id])) if record["claims"] else 0
            is_valid = record["claims"] is not None
            note = "" if is_valid and record["first_ok"] else ("needed retry" if is_valid else record["error"][:140])
            found_text = f"{found}/3"
            print(f"{name:<26} {'yes' if record['first_ok'] else 'NO':>15} {'yes' if is_valid else 'NO':>5} "
                  f"{found_text:>6} {record['tokens_out']:>10} {record['secs']:>7.0f}")
            if note:  # the reason goes on its own lines so the table stays narrow
                print(textwrap.fill(note, 76, initial_indent="    ", subsequent_indent="    "))
            rows.append({"paper": arxiv_id, "method": name, "valid_first": record["first_ok"], "valid": is_valid, "found": found,
                         "tokens_out": record["tokens_out"], "secs": record["secs"], "tries": record["tries"],
                         "error": record["error"]})
    print(f"\nall {len(papers)} papers")
    print(HEAD)
    for name, _ in METHODS:
        method_rows = [row for row in rows if row["method"] == name]
        paper_count = len(method_rows)
        first = f"{sum(row['valid_first'] for row in method_rows)}/{paper_count}"
        valid = f"{sum(row['valid'] for row in method_rows)}/{paper_count}"
        found_text = f"{sum(row['found'] for row in method_rows)}/{3 * paper_count}"
        print(f"{name:<26} {first:>15} {valid:>5} {found_text:>6} "
              f"{sum(row['tokens_out'] for row in method_rows):>10} {sum(row['secs'] for row in method_rows):>7.0f}")
    (HERE / "results").mkdir(exist_ok=True)
    (HERE / "results" / "07_langchain_structured_output.json").write_text(
        json.dumps({"model": MODEL, "think": False, "rows": rows}, indent=1))


paper 2310.13548
method                     valid first try valid  found tokens out seconds
hand-written                           yes   yes    3/3        413      11


LangChain json_schema                  yes   yes    3/3        430       8
LangChain function_calling              NO    NO    0/3        446      11
    LangChain gave None and no error: the model made 0 tool call(s) and
    wrote: '{ "claims": [ { "claim": "Five AI assistants consistently exhibi

paper 2408.02442
method                     valid first try valid  found tokens out seconds
hand-written                           yes   yes    2/3        211       4
LangChain json_schema                  yes   yes    2/3        206       4
LangChain function_calling              NO    NO    0/3        165       3
    LangChain gave None and no error: the model made 0 tool call(s) and
    wrote: '{ "claims": [ { "claim": "LLMs exhibit a significant decline in

paper 2501.10868
method                     valid first try valid  found tokens out seconds
hand-written                           yes   yes    3/3        417       8
LangChain json_schema                  yes   yes    3/3        436 